# Smart Semsar — furniture photo → 3D model server

سيرفر بيحوّل صورة قطعة فرش واحدة لموديل 3D (GLB) علشان تتحط في الـ walkthrough.

**What it does:** runs [TripoSR](https://github.com/VAST-AI-Research/TripoSR) on Colab's GPU and exposes one endpoint through ngrok:

| Endpoint | In | Out |
|---|---|---|
| `GET /health` | – | model name, GPU, whether the model is loaded |
| `POST /generate` | one photo of **one** piece of furniture | a `.glb` file |

The project calls it from `backend/services/colab_service.py`.

**Before you run it**

1. `Runtime → Change runtime type → T4 GPU`.
2. Get a free token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Open the key icon (Secrets) on the left and add two secrets, with "Notebook access" on:
   - `NGROK_TOKEN` — the ngrok token
   - `SMARTSEMSAR_KEY` — any password you choose; the project must send the same one
4. `Runtime → Run all`. The first run takes about 5–10 minutes (it compiles one dependency and downloads the model).
5. Copy the two lines printed by the last cell into the project's `.env`.

The address changes every time the notebook restarts, and a free Colab session stops after a while of inactivity.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
DEVICE = "cuda:0"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install TripoSR and the server packages
Takes a few minutes the first time (`torchmcubes` is compiled here).

In [ ]:
import os

if not os.path.isdir("/content/TripoSR"):
    !git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
%cd /content/TripoSR
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio onnxruntime

## 3. Load the model

In [ ]:
import sys
sys.path.insert(0, "/content/TripoSR")

import rembg
from tsr.system import TSR

model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(8192)      # lower this (e.g. 4096) if the GPU runs out of memory
model.to(DEVICE)
rembg_session = rembg.new_session()
print("model loaded")

## 4. Photo → GLB

- The background is removed, so the photo should show **one** piece, fully visible.
- The model has no real size and may face any way: the project scales and turns it afterwards
  (`backend/services/custom_furniture.py`).
- `mc_resolution` sets the mesh detail: 192 is a good balance, 256 is finer and several MB larger.

In [ ]:
import io
import time

import numpy as np
import trimesh
from PIL import Image
from tsr.utils import remove_background, resize_foreground

MAX_IMAGE_PX = 1024


def photo_to_glb(image_bytes: bytes, mc_resolution: int = 192, foreground_ratio: float = 0.85,
                 strip_background: bool = True) -> bytes:
    image = Image.open(io.BytesIO(image_bytes))
    image.thumbnail((MAX_IMAGE_PX, MAX_IMAGE_PX))
    if strip_background:
        image = remove_background(image.convert("RGB"), rembg_session)
    else:
        image = image.convert("RGBA")
    image = resize_foreground(image, foreground_ratio)
    rgba = np.array(image).astype(np.float32) / 255.0
    rgb = rgba[:, :, :3] * rgba[:, :, 3:4] + (1 - rgba[:, :, 3:4]) * 0.5      # grey behind the object
    image = Image.fromarray((rgb * 255.0).astype(np.uint8))

    with torch.no_grad():
        scene_codes = model([image], device=DEVICE)
    try:        # newer TripoSR: extract_mesh(scene_codes, has_vertex_color, resolution=...)
        mesh = model.extract_mesh(scene_codes, True, resolution=mc_resolution)[0]
    except TypeError:   # older TripoSR: extract_mesh(scene_codes, resolution=...)
        mesh = model.extract_mesh(scene_codes, resolution=mc_resolution)[0]

    # TripoSR's own frame -> glTF's (y up), same turn its demo applies before showing a model
    mesh.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
    mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 1, 0]))
    return mesh.export(file_type="glb")


# quick self-test on TripoSR's own example photo
t0 = time.time()
with open("/content/TripoSR/examples/chair.png", "rb") as f:
    test_glb = photo_to_glb(f.read())
assert test_glb[:4] == b"glTF"
print(f"self-test ok: {len(test_glb) / 1e6:.1f} MB in {time.time() - t0:.0f} s")

## 5. The server

Every request must carry the header `X-API-Key` with your `SMARTSEMSAR_KEY`, so strangers who find the ngrok address cannot use your GPU.

In [ ]:
import threading

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from google.colab import userdata

API_KEY = userdata.get("SMARTSEMSAR_KEY")
assert API_KEY, "Add the secret SMARTSEMSAR_KEY (key icon on the left) and turn on notebook access."

app = FastAPI(title="Smart Semsar furniture 3D")
gpu_lock = threading.Lock()     # one generation at a time


def check_key(x_api_key):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Wrong or missing X-API-Key.")


@app.get("/health")
def health(x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return {"status": "ok", "model": "TripoSR", "gpu": torch.cuda.get_device_name(0)}


@app.post("/generate")
def generate(image: UploadFile = File(...), mc_resolution: int = Form(192),
             strip_background: bool = Form(True), x_api_key: str = Header(default="")):
    check_key(x_api_key)
    if not 64 <= mc_resolution <= 320:
        raise HTTPException(status_code=422, detail="mc_resolution must be between 64 and 320.")
    data = image.file.read()
    try:
        with gpu_lock:
            glb = photo_to_glb(data, mc_resolution=mc_resolution, strip_background=strip_background)
    except Exception as e:      # bad image, out of memory, ...
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")
    return Response(content=glb, media_type="model/gltf-binary")

## 6. Start it and get the public address
Leave this cell running. Stop it to shut the server down.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
assert NGROK_TOKEN, "Add the secret NGROK_TOKEN (key icon on the left) and turn on notebook access."
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("Put these two lines in the project's .env file:\n")
print(f"SMARTSEMSAR_COLAB_URL={public_url}")
print("SMARTSEMSAR_COLAB_KEY=<the SMARTSEMSAR_KEY secret you chose>\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")